# 🧭 **General Workflow for a Regression Task (Template Notebook)**

This notebook is the **template workflow** we use throughout the course for regression problems with `scikit-learn`. All regression labs in this repository follow roughly the same structure:

1. **Import Libraries**
2. **Load the Data & describe the Dataset/Task**
3. **Exploratory Data Analysis (EDA)**
4. **Preprocess the Data** — *split first, then standardize*
5. **Train the Model**
6. **Evaluate the Model** (compare with a baseline)
7. **Compare Methods**

> ⚠️ **Golden rule of preprocessing:** Standardization (or any scaling/encoding that is fit from data) must be performed **after** the train/test split. Fitting the scaler on the full dataset leaks information from the test fold into training through the mean and standard deviation.

# 📈 **ML Regression for Advertising Sales**

---
In this lab, you'll learn how to preprocess tabular data and solve a regression problem using **three equivalent techniques**:

1. **Normal Equation** (closed-form solution with matrix inversion)
2. **QR Decomposition** (numerically stable factorisation)
3. **Scikit-Learn `LinearRegression`** (the library way)

All three produce the same least-squares coefficients. The goal is to understand **why** they are equivalent and **when** you might prefer one over another.

---

# 📊 **Dataset & Task**

**Dataset:** [Advertising dataset](https://www.kaggle.com/datasets/bumba5341/advertisingcsv) — 200 observations of advertising budgets (in thousands of dollars) spent across three channels for a single product, with the corresponding sales (in thousands of units).

| Column | Description | Type |
|--------|-------------|------|
| `TV` | Advertising budget on TV | numeric feature |
| `Radio` | Advertising budget on Radio | numeric feature |
| `Newspaper` | Advertising budget on Newspaper | numeric feature |
| `Sales` | Product sales | **target** (continuous) |

**Task:** Predict the continuous target `Sales` from the three advertising budgets. This is a **regression** problem, and we will solve it using three equivalent techniques: the **Normal Equation**, **QR Decomposition**, and **scikit-learn's `LinearRegression`**. We evaluate with **MSE / RMSE / R²**.

# 1️⃣ Import Libraries


> First install the extra dependencies (only needed the first time in a fresh environment).

In [ ]:
from IPython.display import clear_output

%pip install kagglehub catboost lightgbm tqdm -q

clear_output()

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import kagglehub
import os
from tqdm import tqdm

%matplotlib inline

# 2️⃣ Read the Data


In [ ]:
# Download latest version
path = kagglehub.dataset_download("bumba5341/advertisingcsv")

print("Path to dataset files:", path)

In [ ]:
csv_path = os.path.join(path, "Advertising.csv")

df = pd.read_csv(csv_path)
df = df.drop(columns="Unnamed: 0", axis=1)
df.head()

In [ ]:
df.info()

# 3️⃣ Exploratory Data Analysis (EDA)

**Rule of thumb checklist:**

| Question | If YES | If NO |
|----------|--------|-------|
|  **Is the target skewed?** | Consider log transform | Proceed |
|  **Missing values?** | Impute or drop | Proceed |
|  **Categorical columns?** | Encode them | Proceed |
|  **Duplicates?** | Drop them | Proceed |
|  **Different scales?** | Standardize  | Proceed |


In [ ]:
# 1. What does our target variable (charges) look like?
def check_target_distribution(df, target_column):
  df[target_column].hist(bins=30, edgecolor='black')

  plt.title(f"Target Distribution ({target_column})")
  plt.xlabel(target_column)
  plt.ylabel("Frequency")
  plt.grid(False)

  plt.show()

check_target_distribution(df, "Sales")

> **Rule of thumb:** For regression, if the target is heavily skewed, we might apply a log transform. However, for this lab we'll work with the raw values.


In [ ]:
# 2. Do we have missing values?
def check_missing_values(df):
  missing_values = df.isnull().sum()
  print("Missing Values per Column:")
  print(missing_values[missing_values > 0])
  if missing_values.any():
    print("\nHandle Missing Values as needed.")
  else:
    print("\nNo Missing Values Found.")

check_missing_values(df)

In [ ]:
# 3. Do we have categorical columns?
categorical_cols = df.select_dtypes(include=["object"]).columns

print("Categorical Columns:", list(categorical_cols))

> We don't have categorical columns. If we did, we would need to encode them (convert them into numbers) for our models.

In [ ]:
# 4. Do we have duplicate samples?
def check_duplicates(df):
  duplicates = df.duplicated().sum()
  print(f"Number of Duplicate Samples: {duplicates}")
  if duplicates > 0:
    print("Dropping Duplicates...")
    df.drop_duplicates(inplace=True)
    print("Duplicates Dropped.")
  else:
    print("No Duplicate Samples Found.")

check_duplicates(df)

> **Rule of thumb:** If duplicates exist, drop them with `df.drop_duplicates(inplace=True)`


In [ ]:
# 5. Do we have different scales in the data?
df.describe()

> **Rule of thumb:** If features have vastly different ranges, then **scale them**. Scaling won't hurt, and it's recommended regardless.
>
> Tree models (RF, LightGBM, CatBoost) don't need scaling, but Linear Regression, Ridge, LASSO, SVM do!
>
> ⚠️ **We do NOT scale here.** Scaling must be fit on the **training set only** and then applied to the test set. We will scale **after** the split to avoid data leakage.

In [ ]:
from sklearn.preprocessing import MinMaxScaler

# We only IMPORT the scaler here. We will fit it on X_train inside the K-Fold loop.
# ❌ DO NOT DO THIS (data leakage):
#     scaler = MinMaxScaler().fit(df[features])
#     df[features] = scaler.transform(df[features])
#
# ✅ Instead: split → fit scaler on X_train → transform X_test (see below).


# 4️⃣ Training our Regression Models

> We need to split our data into **X** (features) and **y** (target).


In [ ]:
X = df.drop("Sales", axis=1).astype(float)
y = df['Sales'].astype(float)

---

# Linear Regression — Theory & Helper Functions

### **Linear Regression Model**

Linear regression predicts a continuous target value by finding the best linear relationship between features and target:

$$\hat{y} = X \cdot \theta$$

Where:
- $X$ = feature matrix (with bias term)
- $\theta$ = weight vector (parameters to learn)
- $\hat{y}$ = predicted values

---

### **Mean Squared Error Loss (MSE)**

We use **Mean Squared Error** to measure how well our predictions match the true values:

$$J(\theta) = \frac{1}{2m} \sum_{i=1}^{m} (\hat{y}_i - y_i)^2$$

Where:
- $m$ = number of samples
- $y_i$ = true value
- $\hat{y}_i$ = predicted value

> **Intuition:** If we predict the exact value, MSE is 0. The bigger the difference, the higher the penalty (and it's squared, so large errors are penalized heavily)!

---

### **Normal Equation**

The closed-form solution sets the gradient of the loss to zero and solves directly:

$$\theta = (X^T X)^{-1} X^T y$$

This gives us the optimal weights in **one shot** — no iterative optimisation needed.

> ⚠️ **Caveat:** Computing $(X^T X)^{-1}$ can be numerically unstable when features are correlated or the matrix is ill-conditioned. That's where QR decomposition helps (see Part B).

In [ ]:
# Mean Squared Error in NumPy
def mean_squared_error(y, y_hat):
  return (1 / (2 * len(y))) * np.sum((y_hat - y) ** 2)

In [ ]:
def normal_equation(X, y):
   # Add bias term (intercept)
    m = X.shape[0]
    X_b = np.c_[np.ones((m, 1)), X]

    # Closed-form solution using pseudoinverse
    theta = np.linalg.pinv(X_b.T @ X_b) @ X_b.T @ y

    return theta

In [ ]:
def qr_solve(X, y):
    """Solve least-squares via QR decomposition (with bias column)."""
    m = X.shape[0]
    X_b = np.c_[np.ones((m, 1)), X]  # Add bias term
    Q, R = np.linalg.qr(X_b)
    theta = np.linalg.solve(R, Q.T @ y)
    return theta

# 4️⃣ Split & Scale the Data

> We split **first**, then scale. The scaler is fit on the training set only.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error as sklearn_mse, r2_score

# 80/20 split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, shuffle=True
)

print("X_train:", X_train.shape, " X_test:", X_test.shape)

In [ ]:
# Scale after split: fit on train only
scaler = MinMaxScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

---

# Part A: Normal Equation (Closed-Form Solution)

We solve directly using the formula:

$$\theta = (X^T X)^{-1} X^T y$$

In [ ]:
# Train using the Normal Equation
theta_ne = normal_equation(X_train_scaled, y_train.values)

# Predict (need to add bias column)
X_train_b = np.c_[np.ones((X_train_scaled.shape[0], 1)), X_train_scaled]
X_test_b  = np.c_[np.ones((X_test_scaled.shape[0], 1)),  X_test_scaled]

y_train_pred_ne = X_train_b @ theta_ne
y_test_pred_ne  = X_test_b  @ theta_ne

# Evaluate
ne_train_mse  = sklearn_mse(y_train, y_train_pred_ne)
ne_test_mse   = sklearn_mse(y_test,  y_test_pred_ne)
ne_train_rmse = np.sqrt(ne_train_mse)
ne_test_rmse  = np.sqrt(ne_test_mse)
ne_train_r2   = r2_score(y_train, y_train_pred_ne)
ne_test_r2    = r2_score(y_test,  y_test_pred_ne)

print("Normal Equation Results")
print(f"  Train MSE: {ne_train_mse:.4f}  | Test MSE: {ne_test_mse:.4f}")
print(f"  Train RMSE: {ne_train_rmse:.4f} | Test RMSE: {ne_test_rmse:.4f}")
print(f"  Train R²: {ne_train_r2:.4f}   | Test R²: {ne_test_r2:.4f}")
print(f"\nLearned coefficients (theta): {theta_ne}")

---

# Part B: QR Decomposition

A numerically more stable approach. We factorise $X = QR$ and solve $R\theta = Q^T y$:

$$\theta = R^{-1} Q^T y$$

This avoids computing $(X^T X)^{-1}$ directly, which can be unstable when features are correlated.

In [ ]:
# Train using QR Decomposition
theta_qr = qr_solve(X_train_scaled, y_train.values)

# Predict (bias column already in X_train_b, X_test_b from Part A)
y_train_pred_qr = X_train_b @ theta_qr
y_test_pred_qr  = X_test_b  @ theta_qr

# Evaluate
qr_train_mse  = sklearn_mse(y_train, y_train_pred_qr)
qr_test_mse   = sklearn_mse(y_test,  y_test_pred_qr)
qr_train_rmse = np.sqrt(qr_train_mse)
qr_test_rmse  = np.sqrt(qr_test_mse)
qr_train_r2   = r2_score(y_train, y_train_pred_qr)
qr_test_r2    = r2_score(y_test,  y_test_pred_qr)

print("QR Decomposition Results")
print(f"  Train MSE: {qr_train_mse:.4f}  | Test MSE: {qr_test_mse:.4f}")
print(f"  Train RMSE: {qr_train_rmse:.4f} | Test RMSE: {qr_test_rmse:.4f}")
print(f"  Train R²: {qr_train_r2:.4f}   | Test R²: {qr_test_r2:.4f}")
print(f"\nLearned coefficients (theta): {theta_qr}")

> **Sanity check:** The coefficients from Normal Equation and QR should be identical (up to floating-point noise).

In [ ]:
print("Are Normal Equation and QR coefficients the same?")
print(np.allclose(theta_ne, theta_qr))

---

# Part C: Scikit-Learn `LinearRegression`

Scikit-learn wraps the same mathematics behind a clean **Instantiate → Fit → Predict** API.

> Under the hood, `LinearRegression` uses a similar decomposition. The advantage is **convenience**: no manual bias column, no manual matrix algebra.

In [ ]:
from sklearn.linear_model import LinearRegression

# Train using sklearn
model = LinearRegression()
model.fit(X_train_scaled, y_train)

# Predict
y_train_pred_sk = model.predict(X_train_scaled)
y_test_pred_sk  = model.predict(X_test_scaled)

# Evaluate
sk_train_mse  = sklearn_mse(y_train, y_train_pred_sk)
sk_test_mse   = sklearn_mse(y_test,  y_test_pred_sk)
sk_train_rmse = np.sqrt(sk_train_mse)
sk_test_rmse  = np.sqrt(sk_test_mse)
sk_train_r2   = r2_score(y_train, y_train_pred_sk)
sk_test_r2    = r2_score(y_test,  y_test_pred_sk)

print("Scikit-Learn LinearRegression Results")
print(f"  Train MSE: {sk_train_mse:.4f}  | Test MSE: {sk_test_mse:.4f}")
print(f"  Train RMSE: {sk_train_rmse:.4f} | Test RMSE: {sk_test_rmse:.4f}")
print(f"  Train R²: {sk_train_r2:.4f}   | Test R²: {sk_test_r2:.4f}")
print(f"\nIntercept: {model.intercept_:.4f}")
print(f"Coefficients: {model.coef_}")

# 5️⃣ Results Comparison

> All three methods solve the **same** ordinary least-squares problem, so their results should be **identical** (up to floating-point precision).

In [ ]:
comparison = pd.DataFrame({
    "Method": ["Normal Equation", "QR Decomposition", "Sklearn LinearRegression"],
    "Train MSE":  [ne_train_mse,  qr_train_mse,  sk_train_mse],
    "Test MSE":   [ne_test_mse,   qr_test_mse,   sk_test_mse],
    "Test RMSE":  [ne_test_rmse,  qr_test_rmse,  sk_test_rmse],
    "Test R²":    [ne_test_r2,    qr_test_r2,    sk_test_r2],
})
comparison = comparison.set_index("Method")
comparison.style.format("{:.4f}").set_caption("Comparison of Three Least-Squares Solvers")

### Key Takeaways

| Method | Pros | Cons |
|--------|------|------|
| **Normal Equation** | Simple formula, easy to understand | Numerically unstable for ill-conditioned $X^T X$; requires explicit bias column |
| **QR Decomposition** | Numerically stable; avoids forming $X^T X$ | Slightly more code; requires explicit bias column |
| **Sklearn `LinearRegression`** | Clean API; handles bias automatically; production-ready | Less educational — hides the math |

> **Bottom line:** Use `LinearRegression` in practice. Understand the Normal Equation and QR for the theory behind it.

In [ ]:
# Plot: Predictions vs Ground Truth (all 3 methods overlap)
plt.figure(figsize=(6, 4))
plt.scatter(y_test, y_test_pred_ne, alpha=0.7, label="Normal Equation")
plt.scatter(y_test, y_test_pred_qr, alpha=0.5, marker="x", label="QR Decomposition")
plt.scatter(y_test, y_test_pred_sk, alpha=0.3, marker="s", label="Sklearn LR")
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', linewidth=2, label="Perfect")
plt.xlabel("Actual Sales (Ground Truth)")
plt.ylabel("Predicted Sales")
plt.title("Predictions vs Ground Truth")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

---

## Are our models good?

> Is RMSE = 1.00 good? How to know if a score is good or not?
>
> We need something to compare against. We need a **Baseline**.
>
> If our models are better than the baseline score, then they are good.

**The simplest baseline we can consider is the mean of the target**.

In [ ]:
# Calculate the baseline predictions (mean of the target)
baseline_pred = np.full_like(y, y.mean())

# Evaluate the baseline
baseline_mse  = sklearn_mse(y, baseline_pred)
baseline_rmse = np.sqrt(baseline_mse)
baseline_r2   = r2_score(y, baseline_pred)

print(f"Baseline MSE (using mean target): {baseline_mse:.4f}")
print(f"Baseline RMSE (using mean target): {baseline_rmse:.4f}")
print(f"Baseline R2 (using mean target): {baseline_r2:.4f}")

> **Our models are way better than this baseline because our MSE is lower than the baseline MSE (variance).**

---

# 6️⃣ K-Fold Cross-Validation

> A single train/test split can be **lucky or unlucky**. K-Fold Cross-Validation gives a more robust estimate by repeating the split K times and averaging the results.

In [ ]:
from sklearn.model_selection import KFold

n_splits = 5
kf = KFold(n_splits=n_splits, shuffle=True, random_state=42)

# Storage for all 3 methods
cv_results = {
    "Normal Equation":  {"mse": [], "rmse": [], "r2": []},
    "QR Decomposition": {"mse": [], "rmse": [], "r2": []},
    "Sklearn LR":       {"mse": [], "rmse": [], "r2": []},
}

for fold_idx, (train_index, test_index) in enumerate(kf.split(X)):
    print(f"\nFold {fold_idx + 1}/{n_splits}")

    # Split
    X_tr, X_te = X.iloc[train_index], X.iloc[test_index]
    y_tr, y_te = y.iloc[train_index], y.iloc[test_index]

    # Scale
    sc = MinMaxScaler()
    X_tr_sc = sc.fit_transform(X_tr)
    X_te_sc = sc.transform(X_te)

    # Bias columns
    X_tr_b = np.c_[np.ones((X_tr_sc.shape[0], 1)), X_tr_sc]
    X_te_b = np.c_[np.ones((X_te_sc.shape[0], 1)), X_te_sc]

    # --- Normal Equation ---
    th_ne = normal_equation(X_tr_sc, y_tr.values)
    y_pred_ne = X_te_b @ th_ne
    mse_ne = sklearn_mse(y_te, y_pred_ne)
    cv_results["Normal Equation"]["mse"].append(mse_ne)
    cv_results["Normal Equation"]["rmse"].append(np.sqrt(mse_ne))
    cv_results["Normal Equation"]["r2"].append(r2_score(y_te, y_pred_ne))

    # --- QR Decomposition ---
    th_qr = qr_solve(X_tr_sc, y_tr.values)
    y_pred_qr = X_te_b @ th_qr
    mse_qr = sklearn_mse(y_te, y_pred_qr)
    cv_results["QR Decomposition"]["mse"].append(mse_qr)
    cv_results["QR Decomposition"]["rmse"].append(np.sqrt(mse_qr))
    cv_results["QR Decomposition"]["r2"].append(r2_score(y_te, y_pred_qr))

    # --- Sklearn LinearRegression ---
    lr = LinearRegression()
    lr.fit(X_tr_sc, y_tr)
    y_pred_sk = lr.predict(X_te_sc)
    mse_sk = sklearn_mse(y_te, y_pred_sk)
    cv_results["Sklearn LR"]["mse"].append(mse_sk)
    cv_results["Sklearn LR"]["rmse"].append(np.sqrt(mse_sk))
    cv_results["Sklearn LR"]["r2"].append(r2_score(y_te, y_pred_sk))

    print(f"  NE  Test MSE: {mse_ne:.4f}  | QR Test MSE: {mse_qr:.4f}  | SK Test MSE: {mse_sk:.4f}")

In [ ]:
# K-Fold comparison table
cv_comparison = pd.DataFrame({
    "Method": list(cv_results.keys()),
    "Avg Test MSE":  [np.mean(cv_results[m]["mse"])  for m in cv_results],
    "Avg Test RMSE": [np.mean(cv_results[m]["rmse"]) for m in cv_results],
    "Avg Test R²":   [np.mean(cv_results[m]["r2"])   for m in cv_results],
})
cv_comparison = cv_comparison.set_index("Method")
print("5-Fold Cross-Validation Results\n")
print(cv_comparison.to_string(float_format="{:.4f}".format))

> As expected, all three methods produce the same K-Fold results — confirming they solve the exact same problem. K-Fold gives us a more reliable estimate than the single split above.

# 🎯 **Exercises — SOLUTIONS**

Below are the completed exercises with all blanks filled in.


### Exercise 1 — Split the data

Complete the call to `train_test_split` so that **20% of the samples** go to the test set, and the split is **reproducible** (use `random_state=42`).

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,        # 20% for test
    random_state=42       # reproducible
)

print("X_train:", X_train.shape, " X_test:", X_test.shape)

### Exercise 2 — Scale **after** the split (no leakage!)

Complete the next cell so that the scaler is **fit on the training set only**, and then **applied** to the test set.

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

# Fit on train, transform train
X_train_scaled = scaler.fit_transform(X_train)

# Use the SAME scaler to transform test
X_test_scaled  = scaler.transform(X_test)

print("mean(X_train_scaled) ≈ 0 ?", X_train_scaled.mean(axis=0).round(3))
print("std (X_train_scaled) ≈ 1 ?", X_train_scaled.std(axis=0).round(3))

### Exercise 3 — Solve with the Normal Equation

Implement the normal-equation solution. Remember to add a bias column of ones to `X_train_scaled`.

In [ ]:
# 1) Add a bias column of ones
X_train_b = np.c_[np.ones((X_train_scaled.shape[0], 1)), X_train_scaled]
X_test_b  = np.c_[np.ones((X_test_scaled.shape[0], 1)),  X_test_scaled]

# 2) Compute theta using the normal equation
theta = np.linalg.inv(X_train_b.T @ X_train_b) @ X_train_b.T @ y_train.values

# 3) Predict on the test set and compute RMSE
y_pred = X_test_b @ theta
rmse = np.sqrt(sklearn_mse(y_test, y_pred))

print(f"Normal Equation test RMSE: {rmse:.4f}")

### Exercise 4 — Solve with QR Decomposition

Use `np.linalg.qr` to factorise `X_train_b`, then solve with `np.linalg.solve`. Compare the resulting theta to the normal equation — they should match.

In [ ]:
# 1) QR decomposition of X_train_b
Q, R = np.linalg.qr(X_train_b)

# 2) Solve R @ theta_qr = Q^T @ y_train
theta_qr = np.linalg.solve(R, Q.T @ y_train.values)

# 3) Check that both solutions match
print("Theta (Normal Eq):", theta)
print("Theta (QR)       :", theta_qr)
print("Are they close?  :", np.allclose(theta, theta_qr))

### **Contribution: Sattam Altwaim** :)